### 🔗 Links and Resources
- [Broadcast Join](https://www.databricks.com/discover/pages/optimize-data-workloads-guide#broadcast-hash)

### 📌 Turning off disk cache

In [0]:
spark.conf.set("spark.databricks.io.cache.enabled", "false")

### 📌 Reading data from the sample 'tpch' schema

In [0]:
spark.read.table("samples.tpch.customer").count()

In [0]:
spark.read.table("samples.tpch.customer").display()

In [0]:
spark.read.table("samples.tpch.nation").display()

### 📌 Broadcast Join

In [0]:
df1 = spark.read.table("samples.tpch.customer")

# make the DataFrame relatively large
df1 = df1.union(df1).union(df1).union(df1).union(df1).union(df1).union(df1).union(df1).union(df1).union(df1)

df2 = spark.read.table("samples.tpch.nation")

df3 = df1.join(df2, df1.c_nationkey == df2.n_nationkey, "inner")

# The explain plan should automatically include a broadcast join due to AQE
df3.explain("extended")

In [0]:
# This will leverage the broadcast join and should be relatively quicker vs the Data Shuffle
df3.write.mode("overwrite").saveAsTable("population_metrics.default.customer_nation_bc")

### 📌 Disable Auto Broadcast Join


In [0]:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")

In [0]:
df1 = spark.read.table("samples.tpch.customer")

df1 = df1.union(df1).union(df1).union(df1).union(df1).union(df1).union(df1).union(df1).union(df1).union(df1)

df2 = spark.read.table("samples.tpch.nation")

df3 = df1.join(df2, df1.c_nationkey == df2.n_nationkey, "inner")

# The explain plan should include a Data Shuffle due to disabled broadcast join
df3.explain("extended")

In [0]:
# This will take longer due to the Data Shuffle
df3.write.mode("overwrite").saveAsTable("population_metrics.default.customer_nation_nbc")

### 📌 Changing the Auto Broadcast threshold

In [0]:
# 100MB = 100 × 1024 × 1024 = 104,857,600 bytes
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "104857600")